# Ch.2 — Multiple Linear Regression

**Goal**: Use all 8 California Housing features to improve MAE from $70k → $55k.

| What | Value |
|------|-------|
| Dataset | California Housing (20,640 districts, 8 features) |
| Ch.1 baseline | ~$70k MAE (single feature: MedInc) |
| This chapter target | ~$55k MAE (all 8 features) |
| Grand Challenge target | <$40k MAE |

## Setup & Data Loading

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# ── Load the California Housing dataset ───────────────────────────────────
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target  # MedHouseVal in $100k units

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"\nFeature statistics:")
X.describe().round(2)

## Feature Correlation Analysis

Before fitting, we need to understand how features relate to each other.
Highly correlated features cause **multicollinearity** — unstable individual weights.

In [ ]:
# ── Feature correlation heatmap ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # Upper triangle mask
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Feature Correlation Matrix\n(Look for |ρ| > 0.7 — collinear pairs)', fontsize=14)
plt.tight_layout()
plt.savefig('img/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️ Collinear pairs (|ρ| > 0.7):")
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            print(f"  {corr.columns[i]} ↔ {corr.columns[j]}: ρ = {corr.iloc[i,j]:.2f}")

## VIF — Variance Inflation Factor

VIF quantifies how much each feature's coefficient variance is inflated by collinearity.
- VIF = 1: no collinearity
- VIF > 5: moderate concern
- VIF > 10: severe — drop or regularize

In [ ]:
# ── VIF computation ───────────────────────────────────────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_vif = np.column_stack([np.ones(X_scaled.shape[0]), X_scaled])

print("Variance Inflation Factor (VIF):")
print("-" * 40)
for i, name in enumerate(housing.feature_names):
    vif = variance_inflation_factor(X_vif, i + 1)
    flag = " ⚠️ HIGH" if vif > 5 else ""
    print(f"  {name:12s}: VIF = {vif:6.1f}{flag}")

## Single-Feature Baseline (Ch.1 Recap)

First, establish the Ch.1 baseline so we can measure the improvement.

In [ ]:
# ── Ch.1 baseline: MedInc only ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Single-feature model
model_1feat = LinearRegression()
model_1feat.fit(X_train[['MedInc']], y_train)
y_pred_1 = model_1feat.predict(X_test[['MedInc']])

mae_1 = mean_absolute_error(y_test, y_pred_1) * 100_000
r2_1 = r2_score(y_test, y_pred_1)
print(f"Ch.1 Baseline (MedInc only):")
print(f"  MAE:  ${mae_1:,.0f}")
print(f"  R²:   {r2_1:.4f}")

## Multi-Feature Model — All 8 Features

Now use ALL features. The key question: how much does adding 7 more features help?

In [ ]:
# ── Multi-feature model: all 8 features ──────────────────────────────────
scaler_train = StandardScaler()
X_train_s = scaler_train.fit_transform(X_train)
X_test_s = scaler_train.transform(X_test)  # Use TRAIN statistics!

model_8feat = LinearRegression()
model_8feat.fit(X_train_s, y_train)
y_pred_8 = model_8feat.predict(X_test_s)

mae_8 = mean_absolute_error(y_test, y_pred_8) * 100_000
r2_8 = r2_score(y_test, y_pred_8)
n, d = X_test_s.shape
adj_r2_8 = 1 - (1 - r2_8) * (n - 1) / (n - d - 1)

print(f"Multi-Feature Model (all 8 features):")
print(f"  MAE:     ${mae_8:,.0f}")
print(f"  R²:      {r2_8:.4f}")
print(f"  Adj R²:  {adj_r2_8:.4f}")
print(f"\nImprovement over Ch.1:")
print(f"  MAE:  ${mae_1:,.0f} → ${mae_8:,.0f}  ({(1 - mae_8/mae_1)*100:.1f}% better)")
print(f"  R²:   {r2_1:.4f} → {r2_8:.4f}  (+{(r2_8 - r2_1)*100:.1f}% variance explained)")

## Feature Importance — Which Features Matter?

After standardization, absolute weight magnitude indicates feature importance.
This only works because all features are on the same scale (mean=0, std=1).

In [ ]:
# ── Feature importance (standardized weights) ────────────────────────────
importance = pd.DataFrame({
    'Feature': housing.feature_names,
    'Weight': model_8feat.coef_,
    'Abs Weight': np.abs(model_8feat.coef_)
}).sort_values('Abs Weight', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d4edda' if w > 0 else '#f8d7da' for w in importance['Weight']]
ax.barh(importance['Feature'], importance['Weight'], color=colors, edgecolor='gray')
ax.set_xlabel('Standardized Weight', fontsize=12)
ax.set_title('Feature Importance (Standardized Weights)\n'
             'Green = positive effect on house value | Red = negative', fontsize=14)
ax.axvline(x=0, color='black', linewidth=0.8)
for i, (feat, w) in enumerate(zip(importance['Feature'], importance['Weight'])):
    ax.text(w + 0.02 * np.sign(w), i, f'{w:+.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('img/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 3 features (by absolute weight):")
for _, row in importance.iloc[-3:].iterrows():
    print(f"  {row['Feature']:12s}: {row['Weight']:+.4f}")

## Residual Analysis

The residual plot reveals whether the linear assumption holds.
- **Random scatter** → linear model is appropriate
- **U-shape or curve** → non-linear relationship (need polynomial features in Ch.3)

In [ ]:
# ── Residual plot ─────────────────────────────────────────────────────────
residuals = (y_test - y_pred_8) * 100_000  # Convert to dollars

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_8 * 100_000, residuals, alpha=0.3, s=10, c='steelblue')
axes[0].axhline(y=0, color='red', linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Predicted Value ($)', fontsize=12)
axes[0].set_ylabel('Residual ($)', fontsize=12)
axes[0].set_title('Residuals vs Predicted\n(Look for patterns — U-shape means non-linearity)', fontsize=12)

# Residual distribution
axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Residual ($)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Residual Distribution\n(Should be centered at 0, roughly symmetric)', fontsize=12)

plt.tight_layout()
plt.savefig('img/residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Residual statistics:")
print(f"  Mean:   ${np.mean(residuals):,.0f}  (should be ≈ $0)")
print(f"  Std:    ${np.std(residuals):,.0f}")
print(f"  Median: ${np.median(residuals):,.0f}")

## Manual Gradient Descent (Vectorized)

Educational implementation: same core loop as Ch.1, but now with 8 features.
The gradient formula generalizes naturally from scalar to vector.

In [ ]:
# ── Gradient descent from scratch (8 features) ───────────────────────────
w = np.zeros(8)
b = 0.0
alpha = 0.01
n_train = len(X_train_s)
history = []

for epoch in range(300):
    # Forward pass (vectorized)
    y_hat = X_train_s @ w + b
    error = y_hat - y_train
    
    # Gradients (vectorized)
    dw = (2 / n_train) * (X_train_s.T @ error)  # (8,)
    db = (2 / n_train) * np.sum(error)            # scalar
    
    # Update
    w -= alpha * dw
    b -= alpha * db
    
    # Track
    mae = np.mean(np.abs(error)) * 100_000
    history.append(mae)
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | MAE: ${mae:,.0f}")

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history, color='steelblue', linewidth=2)
ax.axhline(y=mae_8, color='red', linestyle='--', label=f'sklearn MAE: ${mae_8:,.0f}')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MAE ($)', fontsize=12)
ax.set_title('Manual Gradient Descent Convergence (8 Features)', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('img/gradient_descent_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## Comparison Summary

Side-by-side comparison of Ch.1 (single feature) vs Ch.2 (all features).

In [ ]:
# ── Summary comparison ────────────────────────────────────────────────────
print("═" * 55)
print("  PROGRESS CHECK: Ch.1 vs Ch.2")
print("═" * 55)
print(f"  {'Metric':<20} {'Ch.1 (1 feat)':>15} {'Ch.2 (8 feat)':>15}")
print(f"  {'-'*20} {'-'*15} {'-'*15}")
print(f"  {'MAE':<20} {'${:,.0f}'.format(mae_1):>15} {'${:,.0f}'.format(mae_8):>15}")
print(f"  {'R²':<20} {r2_1:>15.4f} {r2_8:>15.4f}")
print(f"  {'Adj R²':<20} {'N/A':>15} {adj_r2_8:>15.4f}")
print(f"  {'Features':<20} {'MedInc':>15} {'All 8':>15}")
print(f"  {'-'*20} {'-'*15} {'-'*15}")
print(f"  {'Target':<20} {'<$40k MAE':>15} {'<$40k MAE':>15}")
print(f"  {'Gap to target':<20} {'${:,.0f}'.format(mae_1 - 40000):>15} {'${:,.0f}'.format(mae_8 - 40000):>15}")
print("═" * 55)
print("\n→ Next: Ch.3 adds polynomial features to capture non-linear patterns")

## Exercises

1. **Incremental features**: Add features one by one (MedInc → +HouseAge → +AveRooms → ...) and plot MAE vs number of features. Which feature gives the biggest single improvement?
2. **Drop collinear features**: Remove AveBedrms (correlated with AveRooms). Does MAE change? Does weight stability improve?
3. **Normal Equation from scratch**: Implement $\mathbf{w}^* = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ manually and compare to sklearn's result.

In [ ]:
# ── Exercise 1 scaffold: incremental feature addition ─────────────────────
# Add features one-by-one and track MAE
feature_order = ['MedInc', 'Latitude', 'Longitude', 'HouseAge',
                 'AveRooms', 'Population', 'AveOccup', 'AveBedrms']
maes = []

for k in range(1, len(feature_order) + 1):
    feats = feature_order[:k]
    # TODO: fit model with feats, compute MAE, append to maes
    pass

# TODO: plot MAE vs number of features

In [ ]:
# ── Exercise 2 scaffold: drop AveBedrms ───────────────────────────────────
# TODO: fit model WITHOUT AveBedrms, compare MAE and weight stability
pass

In [ ]:
# ── Exercise 3 scaffold: Normal Equation from scratch ─────────────────────
# Implement: w* = (XᵀX)⁻¹ Xᵀy
# Compare to model_8feat.coef_ and model_8feat.intercept_
pass